# 交流最適潮流(AC-OPF)+ 離散無効電力補償 — AC Optimal Power Flow

送電系統運用者(ISO/RTO・給電指令所)は、各発電機の有効/無効出力と各バスの電圧を決め、
需要を満たしながら発電費用を最小化しなければならない。**DC近似**(有効電力のみ・線形)は
混雑管理には十分でも、電圧違反・無効電力不足・送電損失は**真の交流(AC)潮流計算でしか
見えない**。本notebookは他のサンプル(big-M型の整数×連続構造を持つ題材)と対比する題材で、初めて
**真の非凸**(電圧×電圧×三角関数)を含むMINLPを扱う。

各バス $i$ の正味有効/無効注入は電圧の大きさ $V$ と位相角 $\theta$ の**真の非凸関数**:

$$
\begin{aligned}
P_i(V,\theta) &= V_i \sum_j V_j\,(G_{ij}\cos(\theta_i-\theta_j) + B_{ij}\sin(\theta_i-\theta_j)),\\
Q_i(V,\theta) &= V_i \sum_j V_j\,(G_{ij}\sin(\theta_i-\theta_j) - B_{ij}\cos(\theta_i-\theta_j)).
\end{aligned}
$$

離散コンデンサバンクの投入段数 $n_i \in \{0,\dots,5\}$(整数)が無効電力補償を行う——これが
本モデルのMINLP化の核心だが、無効バランスに**線形**に入るため、真の非凸は $V\cdot V\cdot\cos/\sin$ 側に集約される(送電線on/off・離散タップのような、非凸項に整数を
直接掛け込む形はより難しく、今後の拡張課題)。

1. **素朴な定式化** — 極座標形式・epigraph化した二次発電費用・sin/cosの使い方
2. **診断** — `mk.analyze` で観測し、真の非凸がどんな症状を生むかを見る(big-M型の他サンプルとの対比)
3. **診断の中身を見る** — ルートLP緩和の違反・空間分枝木を直接見る
4. **改善を試す** — 真の非凸ゆえ「最適性証明」ではなく「実行可能解の質」を軸にした
   改善(SCIPのヒューリスティクス設定)を試し、効果を正直に測る

題材: `samples/energy_and_microgrid/ac_opf.py`

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum, SCIP_PARAMSETTING

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import ac_opf
from minlpkit.collectors.violation import collect_root_violations, violation_by_type
from minlpkit.collectors.tree import solve_and_collect

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

`build_model("small")` は5バス(実行可能性の確認用、それでも真の非凸)。
発電費用は2次凸(`a+b*Pg+c*Pg^2`)だが、**SCIPは非線形目的を直接扱えない**ため
epigraph変数 `cost_g >= a+b*Pg+c*Pg^2` へ移してある——非線形目的を持つMINLPの定石。

In [2]:
m = ac_opf.build_model("small")
nB, nEdge = m.data["dims"]
n_int = sum(1 for v in m.getVars() if v.vtype() in ("INTEGER", "BINARY"))
n_nonlin = sum(1 for c in m.getConss() if not c.isLinear())
print(f"バス数={nB} 送電線数={nEdge} 発電機バス={len(m.data['gen_buses'])} "
      f"コンデンサ候補バス={len(m.data['cap_buses'])}")
print(f"変数: 整数(コンデンサ段数) {n_int} 個 / 連続(V, theta, Pg, Qg, cost) "
      f"{m.getNVars() - n_int} 個")
print(f"制約: {m.getNConss()} 本(内 非線形 = {n_nonlin} 本 = 有効/無効バランス各バス分)")
print("目的: epigraph 'cost_g >= a + b*Pg + c*Pg^2'(2次凸を線形不等式の下界として表現)")

バス数=5 送電線数=5 発電機バス=2 コンデンサ候補バス=3
変数: 整数(コンデンサ段数) 3 個 / 連続(V, theta, Pg, Qg, cost) 16 個
制約: 12 本(内 非線形 = 12 本 = 有効/無効バランス各バス分)
目的: epigraph 'cost_g >= a + b*Pg + c*Pg^2'(2次凸を線形不等式の下界として表現)


## 2. 診断 (`mk.analyze`)

整数×連続の積・disjunctive構造を持つ他のサンプルとは異なり、この問題は**電圧×電圧×三角関数**という
真の非凸を持つ。small規模でどんな症状が出るかを見る。

In [3]:
report_small = mk.analyze(lambda: ac_opf.build_model("small"), name="acopf-small", time_limit=35)
print(report_small.summary())
print()
print({k: report_small.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "n_stalls", "has_nonlinear",
    "bottleneck_type", "bottleneck_rel_viol", "max_linking_groups", "n_heavy_linking"]})

[acopf-small] 検出症状 3件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)

{'gap': 1.7502979779854078, 'nodes': 3940, 'nsols': 14, 'spatial_share': np.float64(0.597625538102988), 'n_stalls': 2, 'has_nonlinear': True, 'bottleneck_type': 'pbal', 'bottleneck_rel_viol': 0.6285734657560618, 'max_linking_groups': 4, 'n_heavy_linking': 2}


big-M型の他のサンプルは軒並み `nodes=1`(root)で解けていた——ここでは
**`weak_relaxation`(緩和違反集中+空間分枝多数)と`dual_stall`(双対境界停滞)の両方が
発火**する。これが「真の非凸」と「big-M disjunctive」の質的な違いである。参考までに
default規模(14バス、IEEE 14相当)でも同じ時間制限でどうなるかを見る。

In [4]:
report_default = mk.analyze(lambda: ac_opf.build_model("default"), name="acopf-default", time_limit=30)
print({k: report_default.metrics.get(k) for k in ["gap", "nodes", "nsols", "spatial_share"]})
print(f"gap {report_default.metrics.get('gap', 0)*100:.0f}% (30秒、実行可能解"
      f"{report_default.metrics.get('nsols')}個) — バス数が3倍弱になっただけでgapは"
      f"桁違いに悪化する。真の非凸の空間分枝限定法はスケールに極めて敏感。")

{'gap': 8.54296023811119, 'nodes': 828, 'nsols': 5, 'spatial_share': 0.0}
gap 854% (30秒、実行可能解5個) — バス数が3倍弱になっただけでgapは桁違いに悪化する。真の非凸の空間分枝限定法はスケールに極めて敏感。


## 3. 診断の中身を見る

### 3-1. ルートLP緩和の違反 — どこが支配的ボトルネックか

In [5]:
m3 = ac_opf.build_model("small")
vdf = collect_root_violations(m3)
by_type = violation_by_type(vdf)
print(by_type.to_string(index=False))

ctype  mean_rel  max_rel  n
 pbal  0.628573 0.916976  5
 qbal  0.522519 0.613240  5
 cost  0.000000 0.000000  2


In [6]:
top_rows = vdf.sort_values("rel_violation", ascending=False).head(nB)
fig = go.Figure(layout=base_layout(
    "ルートLP緩和での非線形制約 違反量(相対) — pbal(有効電力バランス)が支配的",
    "制約", "相対違反", height=380))
fig.add_trace(go.Bar(x=top_rows["constraint"], y=top_rows["rel_violation"],
    marker_color=[C["warn"] if t.startswith("pbal") else C["s1"] for t in top_rows["ctype"]],
    text=[f"{v:.2f}" for v in top_rows["rel_violation"]], textposition="outside", cliponaxis=False))
show(fig)

線形化(McCormick的な)緩和では有効電力バランス `pbal` の違反が60%を超える
——電圧×電圧×cos/sin という真の双線形+三角関数の積を、線形緩和がほとんど捉えられて
いないことを表す。他のサンプルにある整数×連続の積(McCormick緩和)とは緩和の緩さの桁が違う。

### 3-2. 空間分枝木 — 分枝はどこに集中するか

`solve_and_collect` で分枝ノードを収集し、分枝変数の型(spatial=連続変数への空間分枝
/ integer=コンデンサ段数)で色分けする。

In [7]:
m4 = ac_opf.build_model("small")
tree_df = solve_and_collect(m4, max_nodes=1200, node_limit=1200)
counts = tree_df["kind"].value_counts()
print(counts)
print(f"最大深さ: {tree_df['depth'].max()}")

kind
spatial    629
integer    192
root         1
Name: count, dtype: int64
最大深さ: 20


In [8]:
kind_colors = {"spatial": C["s1"], "integer": C["warn"], "root": C["muted"]}
fig = go.Figure(layout=base_layout(
    "空間分枝木(先頭1200ノード) — spatial(電圧・位相角)が分枝の大半を占める",
    "tidy木レイアウト(x)", "深さ(下ほど深い)", height=440))
for kind, color in kind_colors.items():
    sub = tree_df[tree_df["kind"] == kind]
    if sub.empty:
        continue
    fig.add_trace(go.Scattergl(x=sub["x"], y=sub["y"], mode="markers", name=kind,
        marker=dict(size=4, color=color, opacity=0.7),
        hovertemplate="node%{text}<extra></extra>", text=sub["node"]))
show(fig)

In [9]:
spatial_vars = tree_df.loc[tree_df["kind"] == "spatial", "branch_var"].value_counts().head(8)
integer_vars = tree_df.loc[tree_df["kind"] == "integer", "branch_var"].value_counts().head(8)
print("空間分枝の変数トップ8:")
print(spatial_vars.to_string())
print("整数分枝の変数トップ8:")
print(integer_vars.to_string() if not integer_vars.empty else "(整数分枝なし)")

空間分枝の変数トップ8:
branch_var
t_V_1        140
t_V_2        134
t_theta_3    104
t_V_3        104
t_V_4         85
t_theta_2     36
t_theta_1     14
t_theta_4     10
整数分枝の変数トップ8:
branch_var
t_ncap_2    82
t_ncap_4    56
t_ncap_1    54


空間分枝は電圧 `V_*` と位相角 `theta_*` に集中する——非凸潮流の緩和を締めるために
連続変数を分割している、まさに McCormick/空間分枝限定法の教科書的な挙動。整数分枝
(コンデンサ段数)はごく少数——本モデルは整数決定を無効バランスに線形に入れる設計
(ac_opf.pyのdocstring参照)にしたため、真の難しさが連続側に集約されている。

## 4. 改善を試す — 「最適性証明」ではなく「実行可能解の質」を軸にする

真の非凸MINLPでは、この規模(5バス)ですら30秒でgapが200%近く残る——**厳密最適性の
証明を軸にした改善は非現実的**。実務上重要なのは「限られた時間でどれだけ良い実行可能解
(発電費用)が得られるか」である。SCIPのヒューリスティクス強度(`setHeuristics`)を
変えて、同じ時間予算でのプライマル解の質と個数を比較する。

In [10]:
def measure_heuristics(setting_name, apply_fn, time_limit=30):
    m = ac_opf.build_model("small")
    m.hideOutput()
    if apply_fn is not None:
        apply_fn(m)
    m.setParam("limits/time", time_limit)
    t0 = time.time()
    m.optimize()
    return dict(setting=setting_name, status=str(m.getStatus()), nsols=m.getNSols(),
                primal=m.getPrimalbound() if m.getNSols() > 0 else None,
                dual=m.getDualbound(), gap=m.getGap(), nodes=m.getNNodes(),
                time=time.time() - t0)

results = [
    measure_heuristics("既定", None),
    measure_heuristics("heuristics=aggressive", lambda m: m.setHeuristics(SCIP_PARAMSETTING.AGGRESSIVE)),
]
for r in results:
    print(r)

{'setting': '既定', 'status': 'timelimit', 'nsols': 13, 'primal': 41.057551451361384, 'dual': 13.108671151098187, 'gap': 2.132091039443137, 'nodes': 3142, 'time': 29.56397294998169}
{'setting': 'heuristics=aggressive', 'status': 'timelimit', 'nsols': 21, 'primal': 41.03800243129284, 'dual': 10.326857336051674, 'gap': 2.973910076982155, 'nodes': 2400, 'time': 29.93588376045227}


In [11]:
labels = [r["setting"] for r in results]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("見つけた実行可能解の数", "最良の発電費用(低いほど良い)", "双対境界(高いほど良い)"))
bar_colors = [C["muted"], C["s1"]]
fig.add_trace(go.Bar(x=labels, y=[r["nsols"] for r in results], marker_color=bar_colors,
    text=[r["nsols"] for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=[r["primal"] for r in results], marker_color=bar_colors,
    text=[f"{r['primal']:.3f}" for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=[r["dual"] for r in results], marker_color=bar_colors,
    text=[f"{r['dual']:.2f}" for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=3)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="ヒューリスティクス強化の効果(30秒予算) — 解の個数は増えるが質は同等、双対境界は悪化",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

**正直な検証結果**: ヒューリスティクスを強化すると見つかる実行可能解の個数は
増える(既定→aggressive)が、**最良の発電費用(プライマル解の質)はほぼ変わらない**
——コンデンサ段数と発電出力の組み合わせは、既定のヒューリスティクスでも早い段階で
良い解に到達しており、追加の探索は「似たような質の別解」を増やすだけだった。一方で
**双対境界ははっきり悪化**する(ヒューリスティクスに割く時間の分、分枝によるルート
緩和の締め込みが減るため)。つまりこの規模・時間予算では、ヒューリスティクス強化は
「解の多様性」以上の実務的価値をもたらさず、最適性証明(gap)の観点ではむしろ逆効果
——真の非凸MINLPでは「時間予算をどこに使うか」のトレードオフが、
big-M/disjunctive構造よりもシビアに効くことを示している。

## まとめ

- DC近似では見えない電圧・無効電力・送電損失を扱うAC-OPFは、電圧×電圧×三角関数という
  **真の非凸**を持つ。整数(コンデンサ段数)を無効バランスに線形に入れる設計により、
  非凸性は連続側 $(V, \theta)$ に集約される——2節・3節の空間分枝木がそれを直接裏付ける。
- big-M disjunctive構造を持つ他のサンプル(真の非凸なし)は軒並みroot 1ノードで解けたのに対し、
  この問題は5バスという最小規模でも `weak_relaxation`+`dual_stall` が発火し、
  バス数が3倍弱(14バス)になるだけでgapが桁違いに悪化する——**disjunctive構造と
  真の非凸は、診断上まったく異なる難度のクラスに属する**。
- 真の非凸ゆえ最適性証明を軸にした改善は非現実的で、**実行可能解の質**を軸にした
  検証が適切だった。ヒューリスティクス強化は解の個数を増やすが質は変えず、
  双対境界を犠牲にする——「改善が効かない」ことも含めて正直に測った。

### この診断が対応する実務の意思決定

給電指令所が「この系統規模・この時間予算で、どこまで発電費用を追い込めるか」を
判断する材料になる。厳密最適性の証明が非現実的な規模では、SCIPの設定チューニングより
むしろ**問題の分割(N-1想定の縮約・電圧帯ごとの分解)や、より強い凸緩和(SOC緩和・
QC緩和)への切り替え**が今後の拡張課題として重要になる(ac_opf.py docstring
「残課題」参照)。

関連: [手法ガイド](../../playbook/index.md) /
サンプルカタログ [エネルギー・マイクログリッド](../../samples/energy_and_microgrid.md)